# 06. 결과 표·그림·보고서 초안

봉인된 산출물만 읽는다. 새 추론을 하지 않는다.

- D1 주 표(반복 비율 × 비교군, seed 평균과 범위), bootstrap 구간, 용량 민감도
- D2 test 지표와 false reuse 대 호출 절감 그림
- 요청 순서별 누적 비용·오재사용
- end-to-end 지연 **시뮬레이션**: 로컬 실측 구간 + 가정 왕복 120ms/호출 + (xRouteBench만) 기록 생성 시간
- `report/REPORT-draft.md`와 원문·임베딩을 뺀 `results-for-review.zip`

연산 단가(`pricing.compute_usd_per_hour`)를 설정에 채우면 CPU 시간을 달러로 환산해 표기한다. 비워 두면 CPU 시간만 보고한다.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

RUN_ID = os.environ.get("CACHE_EXP_RUN_ID", "run001")  # 새 계획·재시작이면 새 ID
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LLM_ROUTER = Path("/content/drive/MyDrive/LLM_Router")
    KIT = LLM_ROUTER / "cache-experiment"
    zip_path = KIT / "source/cache-source.zip"
    expected = (KIT / "source/cache-source.sha256").read_text().split()[0]
    assert hashlib.sha256(zip_path.read_bytes()).hexdigest() == expected, "source ZIP SHA mismatch"
    REPO = Path("/content/cache-source")
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
        z.extractall(REPO)
    for name, digest in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
        assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
    HERE = REPO / "ml/notebooks/cache_experiment"
    DATA_ROOT, OUTPUT_ROOT = KIT / "data-root", KIT / "runs"
    ROUTER_DIR = LLM_ROUTER / "transfer-experiments/inputs/experiment-A/models/r0"
    os.environ.setdefault("CACHE_EXP_DRIVE_LOGGER", str(LLM_ROUTER / "retraining-effect/logger/logger-log.parquet"))
    assert sys.version_info[:2] in {(3, 11), (3, 12)}, "런타임 버전 2026.07(Python 3.12)을 선택하세요"
    marker = REPO / ".installed-v2"
    if not marker.exists():
        # Text-only experiment. Colab's preinstalled vision stack does not match
        # torch 2.5.1, and transformers imports timm (which needs torchvision)
        # while resolving Auto classes, so remove all three.
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio", "timm"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.5.1",
                        "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(HERE / "requirements-colab.txt")], check=True)
        # Exercise the Auto-class resolution the notebook uses, in a fresh process,
        # so a preinstalled package clash fails here rather than after the restart.
        probe = ("from transformers import AutoConfig, AutoModel; "
                 "AutoModel.from_config(AutoConfig.from_pretrained('BAAI/bge-large-en-v1.5', "
                 "revision='d4aa6901d3a41ba39fb536a557fa166f842b0e09')); print('transformers ok')")
        subprocess.run([sys.executable, "-c", probe], check=True)
        marker.write_text("ok")
        # Colab already imported its own numpy; mixing it with the pinned one breaks
        # transformers. Restart the runtime (files in /content survive) and run all again.
        print("설치 완료. 런타임을 재시작합니다. 재시작되면 '모두 실행'을 다시 누르세요.")
        os.kill(os.getpid(), 9)
    import numpy
    pinned = next(l.split("==")[1].strip() for l in (HERE / "requirements-colab.txt").read_text().splitlines()
                  if l.startswith("numpy=="))
    assert numpy.__version__ == pinned, (
        f"numpy {numpy.__version__} != {pinned}: 런타임 → 세션 다시 시작 후 '모두 실행'을 다시 누르세요")
else:
    # 로컬: 저장소 루트의 .venv-cache 커널. 입력은 ml/data·ml/artifacts가 있는 체크아웃,
    # 산출물은 Drive 동기화 폴더의 kit runs/ 가 있으면 그곳(Colab과 같은 run을 이어 쓴다).
    HERE = Path.cwd()
    DATA_ROOT = Path(os.environ.get("CACHE_EXP_DATA_ROOT", HERE.parents[2]))
    synced = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/*/LLM_Router/cache-experiment/runs"))
    OUTPUT_ROOT = os.environ.get("CACHE_EXP_OUTPUT_ROOT") or (synced[0] if synced else None)
    ROUTER_DIR = None  # config router.package_path
    found = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/*/LLM_Router/retraining-effect/logger/logger-log.parquet"))
    if found:
        os.environ.setdefault("CACHE_EXP_DRIVE_LOGGER", str(found[0]))
CONFIG = Path(os.environ.get("CACHE_EXP_CONFIG", HERE / "experiment_config.json"))
sys.path.insert(0, str(HERE))

import pandas as pd
import cache_experiment as ce
import models
import pipeline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
run = pipeline.Run(DATA_ROOT, RUN_ID, CONFIG, OUTPUT_ROOT)
print("run directory:", run.root)

In [ ]:
import report
main, d2_metrics, latency_table, e2e = report.build(run)
main.filter(regex="repeat_fraction|^arm$|_mean$").round(4)

In [ ]:
from IPython.display import Image, display
for path in sorted(run.path("report", "figures").glob("*.png")):
    display(Image(filename=str(path)))

In [ ]:
pipeline.bundle(run)